# SIRAP Backend · RT-DETR Ecuador

Notebook exclusivo de **Ecuador**, integrado a partir de `old_SIRAP_backend (1).ipynb`:
**1.3–2.2**, **3.5**, **6.3**, **6.4** y **6.5**. Se adaptaron las rutas a
`SIRAP_USA_ECUADOR` y se conservaron las funciones de normalización, revisión de SAVH,
lista CSV, registros, SRI y extracción del nombre. No se incorpora YOLO/Ultralytics.

**Contenido y orden de ejecución**

- **0:** dependencias, rutas y opciones de ejecución.
- **1.1–1.2:** escenas ecuatorianas COCO para entrenar el detector y separar particiones.
- **1.3:** recortes OCR SAVH Guayaquil; etiquetas de cámara no equivalen a revisión humana.
- **2.1–2.2:** normalización ecuatoriana, referencia corregida, duplicados y propuestas pendientes.
- **3.1:** ajuste opcional RT-DETRv2-S con escenas ecuatorianas y exportación OpenVINO.
- **3.2:** inferencia con la semilla 2 compartida o el candidato ecuatoriano explícitamente seleccionado.
- **3.5–3.6:** lista CSV de esta sesión y regla de decisión de demostración.
- **6.1–6.2:** video de demostración y muestreo limitado a ocho cuadros.
- **6.3:** registros CSV y SRI opcional en segundo plano.
- **6.4:** propietario desde el registro local o consulta pública opcional verificable.
- **6.5:** comprobaciones locales de extracción, sin consultar el sitio.
- **6.6:** ejecución integrada del video, enriquecimiento y exportación local.

**Dos tareas distintas:** SAVH aporta recortes y transcripciones para OCR. El detector
necesita escenas completas con **todas** las placas delimitadas y negativos reales.
No se generan cajas de escenas a partir de recortes OCR ni se usan respuestas SRI o
nombres de propietario como etiquetas de entrenamiento.

El entrenamiento y las consultas externas están **apagados por defecto**. Preparar este
notebook no reentrena ni sustituye el modelo publicado. La aplicación sigue usando la
semilla 2 compartida y detección **0,70** / OCR **0,94**; un candidato ecuatoriano se guarda
aparte. El notebook USA no se modifica.


## 0. Preparación del entorno

En Colab, selecciona GPU si vas a entrenar. La instalación conserva el PyTorch con CUDA
que proporciona Colab. Fuera de Colab necesitas PyTorch compatible con tu equipo.
No se montan ni se sincronizan automáticamente Drive o repositorios privados.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "fast-plate-ocr==1.1.0" "gdown>=5,<6" "openvino>=2025.4,<2027" "python-Levenshtein>=0.27" pandas numpy matplotlib pillow requests opencv-python-headless


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, csv, hashlib, zipfile, re, math, inspect
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import Levenshtein
from IPython.display import display

REPO_ROOT=Path(os.environ.get('SIRAP_REPO_ROOT','/content/SIRAP_USA_ECUADOR'))
if not REPO_ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git',str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
COUNTRY='ecuador'
WORK_DIR=Path(os.environ.get('SIRAP_ECUADOR_WORK_DIR','/content/SIRAP_workspace/ecuador'))
OCR_RESULTS_DIR=WORK_DIR/'ocr'
for directory in (WORK_DIR,OCR_RESULTS_DIR): directory.mkdir(parents=True,exist_ok=True)
SAVH_ZIP_CANDIDATES=[REPO_ROOT/'datasets/ecuador/savh/revision_visual/savh_ground_truth_subset.zip',
                     WORK_DIR/'savh_ground_truth_subset.zip']
REVIEW_CSV_FALLBACK=REPO_ROOT/'datasets/ecuador/savh/ocr/revision_placas_142_leida.csv'
USE_GOOGLE_SHEET=False
REVIEW_SHEET_ID=''  # opcional; la revisión versionada funciona sin autenticar Google
PLATE_PATTERNS=((3,4),(3,3))
AUTHORIZED_LIST_PATH=WORK_DIR/'authorized_plates.csv'
# Lista exclusiva del experimento. Copiar una lista propia aquí es una acción explícita.
# No se importa la lista operativa de la aplicación ni se autoriza una placa por aparecer en SAVH.
DETECTOR_THRESHOLD=0.70
OCR_THRESHOLD=0.94
RUN_TRAIN_ECUADOR=False
RUN_INFERENCE=False
RUN_VIDEO=False
USE_ECUADOR_TRAINED_MODEL=False
print('País:',COUNTRY,'| trabajo:',WORK_DIR,'| detección:',DETECTOR_THRESHOLD,'| OCR:',OCR_THRESHOLD)


## 1. Datos y análisis exploratorio
### 1.1 Escenas ecuatorianas para el detector

Coloca tu exportación COCO en `ECUADOR_COCO_DIR`, con `train/` y `valid/` (o `val/`)
y su `_annotations.coco.json`. Reserva `test/` para la evaluación posterior. Las cajas
incluyen placas pequeñas, del fondo y parcialmente ocultas; una placa ilegible sigue
siendo una placa. Las escenas negativas llevan lista de anotaciones vacía.

Separa vehículos, clips y sesiones entre particiones **antes** del entrenamiento.
La comprobación siguiente detecta archivos idénticos entre particiones y, cuando están
presentes, IDs `vehicle_id`, `clip_id` o `session_id` compartidos. Sin esos IDs no puede
demostrar independencia de vehículos o escenas. Los datos del descargador USA no se
presentan como datos ecuatorianos.


In [ ]:
ECUADOR_COCO_DIR=WORK_DIR/'datasets/detection_coco'
MODEL_OUTPUT_DIR=WORK_DIR/'models/ecuador/rtdetrv2_s/seed_2'
TRAIN_RUN_DIR=WORK_DIR/'runs/ecuador/rtdetrv2_s/seed_2'
TRAINING_MODEL_SOURCE='merve/license-plates-rtdetrv2'
# Para continuar exactamente desde pesos USA se necesita el checkpoint Transformers
# (config.json + model.safetensors), no el IR OpenVINO. Asigna su carpeta aquí si existe.
EPOCHS=30
PATIENCE=8
BATCH_SIZE=8
TRAIN_SEED=2


def validate_ecuador_coco(root):
    from src.train import _annotation_file, CocoPlateDataset
    digests,groups,summary={},{},[]
    for split in ('train','val','test'):
        try: path=_annotation_file(root,split)
        except FileNotFoundError:
            if split=='test': continue
            raise
        payload=json.loads(path.read_text())
        images=payload.get('images',[]);annotations=payload.get('annotations',[])
        if not images: raise ValueError(f'Sin imágenes en {split}')
        by_id={row['id']:row for row in images}
        if len(by_id)!=len(images): raise ValueError(f'IDs de imagen repetidos en {split}')
        dataset=CocoPlateDataset(root,split,None)
        for row in images:
            image_path=dataset._image_path(row['file_name'])
            with Image.open(image_path) as image:
                if image.size!=(int(row['width']),int(row['height'])):
                    raise ValueError(f'Dimensiones COCO incorrectas en {split}: {image_path.name}')
            digest=hashlib.sha256(image_path.read_bytes()).hexdigest()
            if digest in digests and digests[digest]!=split:
                raise ValueError(f'Imagen idéntica compartida entre {digests[digest]} y {split}')
            digests[digest]=split
            for key in ('vehicle_id','clip_id','session_id'):
                if row.get(key) is not None:
                    group=(key,str(row[key]))
                    if group in groups and groups[group]!=split:
                        raise ValueError(f'Grupo {key} compartido entre {groups[group]} y {split}')
                    groups[group]=split
        for ann in annotations:
            image=by_id.get(ann['image_id'])
            if image is None:raise ValueError(f'Caja sin imagen en {split}')
            x,y,w,h=map(float,ann['bbox'])
            if not all(math.isfinite(v) for v in (x,y,w,h)) or min(x,y)<0 or min(w,h)<=0 or x+w>image['width']+1 or y+h>image['height']+1:
                raise ValueError(f'Caja inválida en {split}: {ann.get("id")}')
        if split in ('train','val') and not annotations:
            raise ValueError(f'{split} necesita placas anotadas, además de escenas negativas')
        summary.append({'split':split,'images':len(images),'plates':len(annotations)})
    return pd.DataFrame(summary)


### 1.2 Revisión de particiones

La auditoría de particiones se ejecuta cuando existen los datos. No ejecuta inferencia
sobre el test ni lo utiliza para elegir épocas, pesos o umbrales.


In [ ]:
if ECUADOR_COCO_DIR.exists():
    display(validate_ecuador_coco(ECUADOR_COCO_DIR))
else:
    print('Prepara escenas ecuatorianas COCO en',ECUADOR_COCO_DIR,'antes de activar RUN_TRAIN_ECUADOR.')


### 1.3 Datos de lectura: SAVH Guayaquil

`datasets/ecuador/savh/revision_visual/savh_ground_truth_subset.zip`, versionado en GitHub, tiene 739 recortes de placa de SAVH Guayaquil: 472 en `placa_legible/` y 267 en `placa_no_legible/`, copiados sin cambios de las 2000 imágenes del dataset público (capturas del 7, 8 y 11 de junio de 2026). `ground_truth.csv` toma la placa del nombre de cada archivo.

**Esa placa no es una transcripción manual: es la lectura automática de la cámara Dahua.** Lo muestran sus errores: 142 de las 472 placas legibles (30 %) no siguen el formato ecuatoriano, y en la imagen la placa se lee bien (`GSO-7335` aparece como `GS07335`, `GTN-8737` como `6TN8737`, `PBT-2437` como `JPBT2437`). Además, 73 imágenes son copias idénticas de otras (43 grupos; quedan 666 únicas, 399 de ellas legibles) y las 267 no legibles también traen texto, que no se puede comprobar mirando la imagen.

En esta integración SAVH se usa para extraer los recortes, aplicar la normalización
(2.1) y armar la referencia corregida con duplicados y revisión humana (2.2).
Las imágenes pendientes se muestran en grupos de **máximo ocho**, controlados por
`SAVH_SAMPLE_SIZE`. Solo las transcripciones confirmadas o corregidas por una persona
se exportan como etiquetas revisadas. El afinamiento OCR se puede abordar aparte en
`06_afinamiento_ocr.ipynb`; este notebook prepara sus referencias y el entrenamiento
del detector RT-DETR con escenas COCO distintas. No calcula EMR/CER ni usa etiquetas
sugeridas como si estuvieran confirmadas.

Si falta el ZIP, se informa y se omite SAVH sin impedir preparar los demás bloques.


Los recuentos anteriores describen la revisión histórica del adjunto. La ejecución
actual muestra los recuentos de los archivos realmente cargados. Las etiquetas sugeridas
o de cámara no se convierten en etiquetas humanas confirmadas. Esta sección prepara OCR;
no entrena el detector con recortes de placa.


In [ ]:
# 1.3 · Extrae los recortes de SAVH Guayaquil (una vez por sesión, en el disco local de Colab)
_missing = [name for name in ("WORK_DIR", "OCR_RESULTS_DIR", "SAVH_ZIP_CANDIDATES") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, "
                       "corre primero la sección 0 completa.")

SAVH_DIR = WORK_DIR / "savh_ground_truth_subset"  # disco local de Colab: se extrae en cada sesión
SAVH_SAMPLE_SIZE = 8  # máximo de imágenes visibles de la revisión pendiente

savh_zip = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), None)
if not (SAVH_DIR / "ground_truth.csv").exists() and savh_zip:
    with zipfile.ZipFile(savh_zip) as archive:
        SAVH_DIR.mkdir(parents=True,exist_ok=True)
        for member in archive.infolist():
            destination=(SAVH_DIR/member.filename).resolve()
            if not destination.is_relative_to(SAVH_DIR.resolve()):
                raise ValueError('El ZIP contiene una ruta fuera de la carpeta SAVH')
        archive.extractall(SAVH_DIR)

savh_gt = None
if (SAVH_DIR / "ground_truth.csv").exists():
    savh_gt = pd.read_csv(SAVH_DIR / "ground_truth.csv", dtype=str, keep_default_na=False)
    savh_gt.insert(0, "fila_gt", range(1, len(savh_gt) + 1))  # fila de datos: la misma numeración de la hoja de revisión
    print(f"{len(savh_gt)} recortes en {SAVH_DIR}")
    display(savh_gt["folder"].value_counts())
else:
    print("Aviso: no se encontró savh_ground_truth_subset.zip en el repositorio ni en /content; se salta SAVH.")

## 2. Preprocesamiento

**Imágenes del detector.** Se usan escenas COCO y el `AutoImageProcessor` del checkpoint RT-DETRv2-S; no se aplican aumentos de Ultralytics. En inferencia se usa OpenVINO y el manifiesto de preprocesamiento del modelo. Antes del OCR se recorta cada caja con margen proporcional de 0,06. Las cajas y las transcripciones se tratan como referencias distintas.

**Texto de las placas.** Esta sección prepara lo que se compara con el OCR: la normalización de cualquier lectura al formato ecuatoriano (2.1) y la referencia corregida de SAVH (2.2).

### 2.1 Normalización de placas
El texto de un lector de placas no siempre es una placa válida. Antes de decidir el acceso hace falta encontrar la placa en el texto, corregir confusiones típicas por posición y validar el formato. fast-plate-ocr ya devuelve solo letras y dígitos, casi siempre con forma de placa: en el análisis histórico del adjunto, 398 de sus 399 lecturas de legibles ya tenían un formato válido y la normalización corrigió una sola (`GP1226` → `GPI226`). La misma normalización se aplica también a la lectura de la cámara de SAVH, que sí trae errores de formato, y a lo que escribe a mano el operador en la app.

Las placas ecuatorianas tienen **3 letras + 3 o 4 dígitos** ([El Universo](https://www.eluniverso.com/noticias/2020/06/26/nota/7885368/placas-carros-como-identificarlas/)). En la referencia de SAVH aparecen también lecturas con 2 letras + 4 o 5 dígitos, pero salen de errores de la cámara (`GB01825` es `GBO-1825`), así que solo se aceptan los dos formatos reales.

`normalize_plate` hace tres pasos:

1. **Busca la placa dentro del texto.** Corta el texto en tramos (por espacios, guiones, saltos de línea y otra puntuación), descarta la leyenda `ECUADOR` impresa sobre la placa si viene en el texto, y prueba cada tramo y las uniones de 2 o 3 tramos seguidos (`GBA 1234`).
2. **Corrige por posición.** En las 3 primeras posiciones (letras) cambia un dígito por la letra parecida (0→O, 1→I, 5→S, 6→G, 8→B) y en las siguientes (dígitos) una letra por el dígito parecido (O→0, D→0, Q→0, I→1, Z→2, B→8); `|` se lee como 1 y `$` como S. Gana el candidato que valida con menos cambios, y se aceptan como máximo 2: en SAVH, ninguna lectura que necesitó 3 era correcta.
3. **Valida el formato.** Si ningún candidato valida, la lectura queda inválida y la regla de decisión la manda a verificación manual.

**Formato para decidir, guardar y mostrar.** `format_plate` escribe cada placa válida como en Ecuador: 3 letras, guion y 4 dígitos (`GBA-1234`). A las placas antiguas de 3 dígitos les agrega un 0 al inicio de los números (`GMS-429` → `GMS-0429`), para que todas queden en el formato XXX-0000; por eso `GMS-429` y `GMS-0429` cuentan como la misma placa. Ese es el formato de la lista de autorizados, de la regla de decisión, de los CSV y del video. La evaluación del OCR con SAVH compara las placas sin guion, porque mide la lectura y no el formato.

La celda termina con casos de prueba (lecturas reales de la cámara de SAVH y textos con la leyenda, separadores y símbolos) que deben pasar antes de seguir.


In [ ]:
# 2.1 · Normalización de placas al formato ecuatoriano
CONFUSIONS_TO_LETTER = {"0": "O", "1": "I", "5": "S", "6": "G", "8": "B"}  # dígito leído donde va una letra
CONFUSIONS_TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "Z": "2", "B": "8"}  # letra leída donde va un dígito
SYMBOLS_TO_CHAR = {"|": "1", "$": "S"}  # símbolos que un lector de texto libre puede escribir en lugar de un carácter
SEPARATORS = " \t\r\n-‐‑‒–—_.·•:,;+\"'`*()[]{}<>"  # cortan el texto; cualquier otro símbolo invalida su tramo
MAX_CORRECTIONS = 2       # en SAVH, ninguna lectura que necesitó 3 cambios era correcta
PLATE_HEADER = "ECUADOR"  # leyenda impresa sobre la placa: se descarta si un lector la transcribe junto con la placa
_SEGMENT = re.compile("[^" + re.escape(SEPARATORS) + "]+")  # tramo: caracteres seguidos sin separadores


def clean_plate_text(text) -> str:
    """Mayúsculas y solo letras y dígitos: 'gba-1234' → 'GBA1234'."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


def is_plate_header(segment: str) -> bool:
    """'ECUADOR' y sus lecturas aproximadas ('EQUADOR', 'ECUNDERG'): solo letras y a 3 cambios o menos."""
    return (len(segment) >= 5 and segment.isascii() and segment.isalpha()
            and Levenshtein.distance(segment, PLATE_HEADER) <= 3)


def as_plate(candidate: str, corrections: bool = True):
    """(placa, cambios) si el candidato tiene formato de placa, corrigiendo por posición si hace falta; si no, None."""
    for n_letters, n_digits in PLATE_PATTERNS:
        if len(candidate) != n_letters + n_digits:
            continue
        letters, digits = candidate[:n_letters], candidate[n_letters:]
        if corrections:
            letters = "".join(CONFUSIONS_TO_LETTER.get(c, c) for c in letters)
            digits = "".join(CONFUSIONS_TO_DIGIT.get(c, c) for c in digits)
        plate = letters + digits
        changes = sum(a != b for a, b in zip(plate, candidate))
        if changes <= MAX_CORRECTIONS and re.fullmatch(f"[A-Z]{{{n_letters}}}[0-9]{{{n_digits}}}", plate):
            return plate, changes
    return None


def find_plate(raw_text, search: bool = True, corrections: bool = True) -> tuple:
    """Busca la placa en el texto del OCR → (placa, formato válido, cambios, (inicio, fin) en el texto).

    Prueba cada tramo entre separadores y las uniones de 2 o 3 tramos seguidos ('GBA 1234'), sin la leyenda
    ECUADOR. Gana el candidato que valida con menos cambios y, a igualdad, el que aparece primero. Si ninguno
    valida, devuelve el texto limpio sin la leyenda, marcado como inválido. `search=False` y `corrections=False`
    apagan cada paso, para la ablación."""
    text = str(raw_text).upper()
    if not search:  # el texto entero como un solo candidato
        found = as_plate(clean_plate_text(text), corrections)
        return (found[0], True, found[1], None) if found else (clean_plate_text(text), False, 0, None)
    segments = [(m.start(), m.end(), "".join(SYMBOLS_TO_CHAR.get(c, c) for c in m.group()))
                for m in _SEGMENT.finditer(text)]
    segments = [s for s in segments if not is_plate_header(s[2])]
    best = None
    for i in range(len(segments)):
        for j in range(i + 1, min(i + 3, len(segments)) + 1):
            found = as_plate("".join(s[2] for s in segments[i:j]), corrections)
            if found and (best is None or found[1] < best[2]):
                best = (found[0], True, found[1], (segments[i][0], segments[j - 1][1]))
    return best or (clean_plate_text("".join(s[2] for s in segments)), False, 0, None)


def normalize_plate(raw_text: str) -> tuple[str, bool]:
    """Texto del OCR → (placa normalizada, formato válido)."""
    plate, is_valid, _, _ = find_plate(raw_text)
    return plate, is_valid


def format_plate(text) -> str:
    """Formato de Ecuador para decidir, guardar y mostrar: 3 letras, guion y 4 dígitos ('gba1234' → 'GBA-1234'). A las
    placas antiguas de 3 dígitos se les agrega un 0 al inicio de los números ('GMS-429' → 'GMS-0429'). Un texto que no
    es una placa válida queda limpio (mayúsculas, solo letras y dígitos), sin guion."""
    plate = clean_plate_text(text)
    match = re.fullmatch(r"([A-Z]{3})(\d{3,4})", plate)
    return f"{match.group(1)}-{match.group(2).zfill(4)}" if match else plate


def plate_confidence(text: str, char_probs) -> float:
    """Confianza del OCR en la placa: la menor probabilidad entre sus letras y dígitos dentro del texto
    (sin placa válida, entre todas las letras y dígitos). NaN si no hay probabilidades."""
    if char_probs is None or len(char_probs) != len(text):
        return float("nan")
    span = find_plate(text)[3]
    start, end = span if span and len(text.upper()) == len(text) else (0, len(text))
    probs = [p for c, p in zip(text[start:end], char_probs[start:end]) if c.isalnum() or c in SYMBOLS_TO_CHAR]
    return float(min(probs)) if probs else 0.0


# Casos de prueba: lecturas reales de la cámara en SAVH (revisión de placas) y textos con la leyenda ECUADOR,
# separadores y símbolos. Cada uno debe dar la placa esperada.
for raw, expected in [("GB01825", "GBO1825"), ("GS11034", "GSI1034"), ("6TK6566", "GTK6566"), ("GTE5D45", "GTE5045"),
                      ("GP1986", "GPI986"), ("gba-1234", "GBA1234"), ("ECUADOR\nGMS-429", "GMS429"),
                      ("EQUIDOR\nGTE-7040", "GTE7040"), ("ECUNDERG\nGB0-6445", "GBO6445"), ("G5V-3233", "GSV3233"),
                      ("G50-6164", "GSO6164"), ("G|G-430", "GIG430"), ("GSH+2315", "GSH2315"), ("GTT·4568", "GTT4568"),
                      ("ECUADOR GMS 429", "GMS429")]:
    normalized, is_valid = normalize_plate(raw)
    assert is_valid and normalized == expected, f"{raw!r} → {normalized} (se esperaba {expected})"
# Lecturas que deben quedar inválidas (→ verificación manual) en vez de convertirse en otra placa válida.
for raw in ["GTT-187G", "GPA-1G88", "OBX=660", "GBΦ6184", "115220", "ECUADOR", ""]:
    normalized, is_valid = normalize_plate(raw)
    assert not is_valid, f"{raw!r} no debería validar (salió {normalized})"
assert plate_confidence("ECUADOR\nGMS-429", [0.2] * 8 + [0.9, 0.8, 0.95, 0.3, 0.99, 0.7, 0.9]) == 0.7
for raw, expected in [("gba1234", "GBA-1234"), ("GBA-1234", "GBA-1234"), ("GMS429", "GMS-0429"), ("gms-429", "GMS-0429"),
                      ("PBA 123", "PBA-0123"), ("PBA-0123", "PBA-0123"), ("G8A1234", "G8A1234"), ("", "")]:
    assert format_plate(raw) == expected, f"{raw!r} → {format_plate(raw)!r} (se esperaba {expected!r})"
print("normalize_plate y format_plate: los casos de prueba pasan.")

### 2.2 Referencia corregida de SAVH (revisión de placas)

La hoja de Google **Revisión placas SAVH (142 legibles)**, en el repositorio de GitHub, tiene una fila por cada placa legible cuya lectura de la cámara no sigue el formato ecuatoriano. La columna `sugerencia` la propuso Claude el 16/10/2026 con reglas de formato y una revisión visual de cada imagen: 118 sugerencias coinciden con la imagen, 15 se corrigieron mirándola (la regla sola fallaba, por ejemplo `BX4948` → `PBX4948`) y 9 quedaron dudosas (filas con `n` = 7–9, 20–22 y 71–73 en la hoja). Son propuestas: la placa definitiva es la que confirme una persona mirando la imagen.

**Para confirmar**, en la última columna de la hoja se escribe una de estas respuestas. Basta con una fila por grupo de copias idénticas (columna `igual_a`): la respuesta se aplica a todo el grupo.

| Respuesta en la hoja | Placa que se usa | `estado_gt` |
|---|---|---|
| `ok` | la sugerencia | `confirmada` |
| una placa, p. ej. `GBU-1825` | esa placa | `corregida_por_revisor` |
| `ilegible` | ninguna: se excluye del EMR | `excluida_por_revisor` |
| vacía | la sugerencia, por ahora | `sugerida_sin_confirmar` |

Las filas que se agregaron a la hoja por desacuerdos entre la cámara y el OCR no traen sugerencia: ahí `ok` confirma la lectura de la cámara (`texto_actual`), y mientras no tengan respuesta la referencia sigue siendo esa lectura.

**Ojo con la O y la Q en la tercera letra.** La cámara escribe 0 para las dos y las sugerencias siempre proponen O, pero en la fila con `n` = 74 fast-plate-ocr leyó Q, y la misma placa de la fila 83 aparece en otras dos imágenes como `GSQ8533` según la cámara. Conviene mirar con cuidado las filas 74 y 83–85.

Las demás legibles quedan como `camara_formato_valido` (tienen formato válido y, en una muestra de 16, 15 coincidían con la imagen y 1 era dudosa) y las no legibles como `no_legible`. La ejecución desde GitHub usa la copia CSV versionada del repositorio; no lee Google Sheets ni requiere autenticar Google Drive. La referencia corregida se guarda en `WORK_DIR/ocr/savh_ground_truth_corregido.csv`, junto con una copia de la referencia usada, para que el informe cite datos fijos. Las copias idénticas se marcan para contar cada imagen una vez. Las dos celdas siguientes arman la referencia y muestran las imágenes que todavía no tienen respuesta.

Si la celda se corre sola en una sesión nueva, avisa qué celdas faltan: la sección 0, la extracción de SAVH (1.3) y la normalización (2.1).


In [ ]:
_missing = [name for name in ("OCR_RESULTS_DIR", "REVIEW_SHEET_ID", "REVIEW_CSV_FALLBACK", "PLATE_PATTERNS",
                              "clean_plate_text", "savh_gt", "SAVH_DIR") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, corre "
                       "primero la sección 0 completa, la extracción de SAVH (1.3) y la normalización (2.1).")

# 2.2 · Referencia corregida de SAVH
REVIEW_PLATE_COLUMN = "revision: ok o placa correcta"
REVIEW_OK = {"ok", "si", "sí"}
REVIEW_EXCLUDE = {"ilegible", "no legible", "excluir"}
CORRECTED_GT_PATH = OCR_RESULTS_DIR / "savh_ground_truth_corregido.csv"
REVIEW_SNAPSHOT_PATH = OCR_RESULTS_DIR / "revision_placas_142_leida.csv"  # la hoja tal como se leyó
STATE_PRIORITY = ["corregida_por_revisor", "excluida_por_revisor", "confirmada", "sugerida_sin_confirmar",
                  "camara_formato_valido", "no_legible"]  # dentro de un grupo de copias gana la respuesta más informativa


def has_plate_format(text: str) -> bool:
    """True si el texto (sin guion) es 3 letras + 3 o 4 dígitos (PLATE_PATTERNS)."""
    return any(len(text) == letters + digits and text[:letters].isalpha() and text[letters:].isdigit()
               for letters, digits in PLATE_PATTERNS)


def read_review_sheet() -> pd.DataFrame:
    """La hoja de revisión, leída con la cuenta de Google de Colab o, si no se puede, desde la copia en CSV."""
    if USE_GOOGLE_SHEET and REVIEW_SHEET_ID:
        try:
            from google.colab import auth
            import gspread
            from google.auth import default
            auth.authenticate_user()
            credentials, _ = default()
            values = gspread.authorize(credentials).open_by_key(REVIEW_SHEET_ID).sheet1.get_all_values()
            review = pd.DataFrame(values[1:], columns=values[0])
            print(f"Hoja de revisión leída desde Google Sheets: {len(review)} filas.")
        except Exception as error:
            if not REVIEW_CSV_FALLBACK.exists():
                raise FileNotFoundError(f"no se pudo leer la hoja ({type(error).__name__}) y no existe {REVIEW_CSV_FALLBACK}.") from error
            review = pd.read_csv(REVIEW_CSV_FALLBACK, dtype=str, keep_default_na=False)
            print(f"No se pudo leer Google Sheets ({type(error).__name__}); se usa el CSV versionado.")
    else:
        if not REVIEW_CSV_FALLBACK.exists():
            raise FileNotFoundError(f"no existe el CSV versionado de revisión: {REVIEW_CSV_FALLBACK}")
        review = pd.read_csv(REVIEW_CSV_FALLBACK, dtype=str, keep_default_na=False)
        print(f"Revisión leída desde GitHub: {len(review)} filas.")
    review = review.astype(str).apply(lambda column: column.str.strip())
    missing = {"n", "fila_gt", "imagen", "texto_actual", "sugerencia", "chequeo_visual", REVIEW_PLATE_COLUMN} - set(review.columns)
    if missing:
        raise ValueError(f"a la hoja de revisión le faltan columnas: {sorted(missing)}")
    return review


def build_corrected_reference(gt: pd.DataFrame, review: pd.DataFrame) -> tuple:
    """Aplica la revisión a ground_truth.csv (por fila_gt) y marca las copias idénticas de una misma imagen."""
    ref = gt.rename(columns={"plate_gt": "plate_camara"}).set_index("fila_gt")
    ref["plate_gt"] = ref["plate_camara"]
    ref["estado_gt"] = np.where(ref["folder"] == "legible", "camara_formato_valido", "no_legible")
    ref["n_hoja"], ref["chequeo_visual"], ref["respuesta"] = "", "", ""
    warnings = []
    for rec in review.to_dict("records"):
        row = int(rec["fila_gt"])
        if row not in ref.index or ref.at[row, "plate_camara"] != rec["texto_actual"]:
            warnings.append(f"fila_gt {row} ({rec['texto_actual']}) no coincide con ground_truth.csv; se ignora.")
            continue
        if Path(ref.at[row, "image_path"]).name != Path(rec["imagen"]).name:
            warnings.append(f"fila_gt {row}: el nombre de imagen de la hoja no coincide; se usa fila_gt.")
        answer = rec[REVIEW_PLATE_COLUMN].lower().strip(" .")
        suggestion = clean_plate_text(rec["sugerencia"]) or clean_plate_text(rec["texto_actual"])  # sin sugerencia: la cámara
        if answer in REVIEW_OK:
            plate, state = suggestion, "confirmada"
        elif answer in REVIEW_EXCLUDE:
            plate, state = "", "excluida_por_revisor"
        elif answer:
            plate, state = clean_plate_text(answer), "corregida_por_revisor"
        elif rec["sugerencia"]:
            plate, state = suggestion, "sugerida_sin_confirmar"
        else:  # agregada por desacuerdo con el OCR y todavía sin respuesta: sigue la lectura de la cámara
            plate, state = suggestion, "camara_formato_valido"
        if state != "excluida_por_revisor" and not has_plate_format(plate):
            warnings.append(f"fila_gt {row}: «{plate}» no tiene formato de placa ecuatoriana.")
        ref.loc[row, ["plate_gt", "estado_gt", "n_hoja", "chequeo_visual", "respuesta"]] = [
            plate, state, rec["n"], rec["chequeo_visual"], answer]
    ref = ref.reset_index()

    # Copias idénticas: cada grupo cuenta una vez y todo el grupo toma la respuesta más informativa.
    first_by_digest, group = {}, []
    for row, path in zip(ref["fila_gt"], ref["image_path"]):
        digest = hashlib.sha256((SAVH_DIR / path).read_bytes()).hexdigest()
        group.append(first_by_digest.setdefault(digest, row))
    ref["duplicado_de_fila"] = pd.array([g if g != r else pd.NA for g, r in zip(group, ref["fila_gt"])], dtype="Int64")
    priority = ref["estado_gt"].map({state: i for i, state in enumerate(STATE_PRIORITY)})
    best = (ref.assign(grupo=group, prioridad=priority).sort_values(["grupo", "prioridad"])
            .drop_duplicates("grupo").set_index("grupo"))
    ref["plate_gt"] = [best.at[g, "plate_gt"] for g in group]
    ref["estado_gt"] = [best.at[g, "estado_gt"] for g in group]
    return ref, warnings


savh_ref = None
if savh_gt is not None:
    try:
        review = read_review_sheet()
    except (FileNotFoundError, ValueError) as error:
        print("Aviso: sin revisión de placas,", error)
        print("Se salta la evaluación de SAVH: con la lectura de la cámara como referencia, el EMR no sería confiable.")
    else:
        review.to_csv(REVIEW_SNAPSHOT_PATH, index=False)
        savh_ref, warnings = build_corrected_reference(savh_gt, review)
        for warning in warnings:
            print("Aviso:", warning)
        savh_ref.to_csv(CORRECTED_GT_PATH, index=False)
        unique = savh_ref["duplicado_de_fila"].isna()
        display(pd.crosstab(savh_ref["estado_gt"], unique.map({True: "imágenes únicas", False: "copias"}),
                            margins=True, margins_name="total"))
        excluded = savh_ref["estado_gt"] == "excluida_por_revisor"
        changed = int(((savh_ref["plate_gt"] != savh_ref["plate_camara"]) & unique & ~excluded).sum())
        print(f"{changed} imágenes únicas cambian de placa respecto a la cámara y {int((unique & excluded).sum())} "
              f"quedan excluidas por el revisor. Referencia guardada en {CORRECTED_GT_PATH}")

In [ ]:
# 2.2 · Imágenes de la hoja de revisión que todavía no tienen respuesta
SHOW_PENDING_REVIEW = True  # imágenes de la hoja que todavía no tienen respuesta, una por grupo de copias

if SHOW_PENDING_REVIEW and savh_ref is not None:
    unique = savh_ref["duplicado_de_fila"].isna()
    added = (savh_ref["estado_gt"] == "camara_formato_valido") & (savh_ref["n_hoja"] != "")  # desacuerdos sin respuesta
    pending = savh_ref[unique & ((savh_ref["estado_gt"] == "sugerida_sin_confirmar") | added)]
    if pending.empty:
        print("Todas las placas de la hoja de revisión tienen respuesta.")
    else:
        pending = pending.assign(orden=pd.to_numeric(pending["n_hoja"], errors="coerce")).sort_values("orden")
        print(f"{len(pending)} imágenes por confirmar en la hoja de revisión (el número es la columna n de la hoja; en rojo, "
              "las dudosas; en azul, los desacuerdos con el OCR, con la lectura de la cámara).")
        pending = pending.head(min(8, int(SAVH_SAMPLE_SIZE or 8)))  # máximo ocho imágenes visibles por revisión
        columns = 4
        rows = -(-len(pending) // columns)
        fig, axes = plt.subplots(rows, columns, figsize=(2.7 * columns, 1.7 * rows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, rec in zip(axes.flat, pending.itertuples(index=False)):
            ax.imshow(Image.open(SAVH_DIR / rec.image_path).convert("RGB"))
            doubtful = rec.chequeo_visual == "dudosa"
            disagreement = rec.estado_gt == "camara_formato_valido"
            tag = " · dudosa" if doubtful else (" · desacuerdo" if disagreement else "")
            ax.set_title(f"n {rec.n_hoja} · {rec.plate_gt}{tag}", fontsize=8.5,
                         color="#b91c1c" if doubtful else ("#1d4ed8" if disagreement else "#0b0b0b"))
        plt.tight_layout()
        plt.show()

In [ ]:
# Etiquetas revisadas para una futura partición OCR; no son escenas para el detector.
# No mezclar estos recortes con un test reservado: separar por vehículo/sesión y hash.
ocr_reviewed=pd.DataFrame()
if savh_ref is not None:
    accepted=savh_ref['estado_gt'].isin(['confirmada','corregida_por_revisor'])
    ocr_reviewed=savh_ref[accepted & savh_ref['duplicado_de_fila'].isna()].copy()
    ocr_reviewed=ocr_reviewed[ocr_reviewed['plate_gt'].map(has_plate_format)]
    ocr_reviewed.to_csv(OCR_RESULTS_DIR/'ecuador_ocr_etiquetas_revisadas.csv',index=False)
    print('Recortes únicos con transcripción confirmada:',len(ocr_reviewed))


## 3. Modelado
### 3.1 Entrenar RT-DETRv2-S con escenas ecuatorianas (opcional)

Con los datos COCO revisados, activa `RUN_TRAIN_ECUADOR=True` y vuelve a ejecutar esta
celda. Máximo 30 épocas, paciencia 8, semilla 2; selecciona el checkpoint con menor
pérdida en validación. El test no interviene. Guarda Transformers y OpenVINO en
`WORK_DIR/models/ecuador/…`, sin cambiar el checkpoint privado USA ni el de la aplicación.

El IR OpenVINO sirve para inferir; para continuar un entrenamiento USA se necesitan
sus pesos Transformers originales. Si `TRAINING_MODEL_SOURCE` sigue con su valor inicial,
el ajuste empieza en el checkpoint general de placas indicado, no en los pesos USA.
La celda adapta los argumentos de Transformers a la versión instalada y no depende
incondicionalmente de `warmup_ratio` o `save_safetensors`.


In [ ]:
def training_arguments_ecuador(TrainingArguments,total_steps):
    supported=inspect.signature(TrainingArguments).parameters
    values=dict(output_dir=str(TRAIN_RUN_DIR),num_train_epochs=EPOCHS,
        learning_rate=5e-5,weight_decay=1e-4,per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,save_strategy='epoch',save_total_limit=2,
        load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,
        prediction_loss_only=True,  # validación por pérdida, sin concatenar etiquetas COCO de tamaños distintos
        remove_unused_columns=False,dataloader_num_workers=2,seed=TRAIN_SEED,
        data_seed=TRAIN_SEED,report_to='none',logging_steps=25)
    values['eval_strategy' if 'eval_strategy' in supported else 'evaluation_strategy']='epoch'
    if 'warmup_ratio' in supported: values['warmup_ratio']=0.05
    elif 'warmup_steps' in supported: values['warmup_steps']=max(1,math.ceil(total_steps*0.05))
    if 'save_safetensors' in supported: values['save_safetensors']=True
    return values

if RUN_TRAIN_ECUADOR:
    import torch
    from transformers import AutoImageProcessor,RTDetrV2ForObjectDetection,TrainingArguments,Trainer,EarlyStoppingCallback
    from src.train import CocoPlateDataset,detection_collator
    from src.export_openvino import export_openvino_model
    display(validate_ecuador_coco(ECUADOR_COCO_DIR))
    if (MODEL_OUTPUT_DIR/'openvino/openvino_model.xml').exists():
        raise FileExistsError('Ya existe un candidato Ecuador. Usa otra MODEL_OUTPUT_DIR y TRAIN_RUN_DIR para una nueva corrida.')
    torch.manual_seed(TRAIN_SEED)
    processor=AutoImageProcessor.from_pretrained(str(TRAINING_MODEL_SOURCE),trust_remote_code=False,use_fast=False)
    model=RTDetrV2ForObjectDetection.from_pretrained(str(TRAINING_MODEL_SOURCE),trust_remote_code=False)
    if model.config.num_labels!=1: raise ValueError('El checkpoint inicial debe tener una sola clase de placa.')
    model.config.id2label={0:'license_plate'};model.config.label2id={'license_plate':0}
    train_data=CocoPlateDataset(ECUADOR_COCO_DIR,'train',processor)
    val_data=CocoPlateDataset(ECUADOR_COCO_DIR,'val',processor)
    total_steps=math.ceil(len(train_data)/BATCH_SIZE)*EPOCHS
    options=training_arguments_ecuador(TrainingArguments,total_steps)
    options['fp16']=torch.cuda.is_available()
    args=TrainingArguments(**options)
    trainer=Trainer(model=model,args=args,train_dataset=train_data,eval_dataset=val_data,
        data_collator=partial(detection_collator,image_processor=processor),processing_class=processor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    transformers_dir=MODEL_OUTPUT_DIR/'transformers'
    trainer.save_model(str(transformers_dir));processor.save_pretrained(str(transformers_dir))
    manifest={'country':'ecuador','architecture':'RT-DETRv2-S','seed':TRAIN_SEED,
        'training_source':str(TRAINING_MODEL_SOURCE),'dataset_dir':str(ECUADOR_COCO_DIR),
        'epochs_requested':EPOCHS,'early_stopping_patience':PATIENCE,
        'checkpoint_selection':'validation eval_loss','best_checkpoint':trainer.state.best_model_checkpoint,
        'detector_threshold':DETECTOR_THRESHOLD,'ocr_threshold':OCR_THRESHOLD,'independent_test_evaluated':False}
    openvino_dir=export_openvino_model(transformers_dir,MODEL_OUTPUT_DIR/'openvino',image_size=640)
    for directory in (transformers_dir,openvino_dir):
        (directory/'sirap_training.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Candidato Ecuador guardado en',MODEL_OUTPUT_DIR,'; no se publica ni activa automáticamente.')
else:
    print('Entrenamiento preparado. RUN_TRAIN_ECUADOR=False: no se entrenó ni se sustituyeron pesos.')


### 3.2 Carga del modelo e inferencia

`RUN_INFERENCE=True` carga el modelo. Por defecto reproduce el checkpoint compartido
USA semilla 2 con reglas ecuatorianas, usando `SIRAP_USA_MODEL_TOKEN` desde el entorno
o Secrets de Colab para leer el modelo privado. No pegues el secreto en el notebook.

Para probar el candidato local generado en 3.1, establece
`USE_ECUADOR_TRAINED_MODEL=True`. Solo cambia la inferencia de **este notebook**, no la
aplicación desplegada. Mantiene detección 0,70 y OCR global 0,94. Las probabilidades
no reemplazan una evaluación independiente con placas ecuatorianas.


In [ ]:
pipeline=None
if RUN_INFERENCE:
    from src.model import PlateDetector,PlateReader
    from src.usa_pipeline import USAPipeline
    from src.utils import load_config,get_secret
    if USE_ECUADOR_TRAINED_MODEL:
        ir_dir=MODEL_OUTPUT_DIR/'openvino'
        manifest=json.loads((ir_dir/'sirap_training.json').read_text())
        if manifest.get('country')!='ecuador': raise ValueError('El candidato no está identificado como Ecuador.')
        detector=PlateDetector(ir_dir,conf=DETECTOR_THRESHOLD,imgsz=640,device='CPU',fallback_checkpoint=None)
        reader=PlateReader('cct-s-v2-global-model','cpu')
        pipeline=USAPipeline(detector,reader,country='ecuador')
    else:
        secret=get_secret('SIRAP_USA_MODEL_TOKEN')
        if secret:os.environ['SIRAP_USA_MODEL_TOKEN']=secret
        pipeline=USAPipeline.from_config(load_config('config/countries/usa.yaml')).for_country('ecuador',PLATE_PATTERNS)
    print('Pipeline:',COUNTRY,pipeline.detector.name,'| candidato local:',USE_ECUADOR_TRAINED_MODEL)
else:
    print('Inferencia desactivada. Activa RUN_INFERENCE cuando quieras cargar el modelo.')


### 3.5 Lista de placas autorizadas

Un CSV con una placa por línea local en `WORK_DIR/authorized_plates.csv`. Si no existe se crea vacío para que el pipeline no falle. **La lista autorizada se mantiene en este CSV**, una placa por línea. Las placas cercanas a una autorización pasan a verificación manual. La lista simulada con la que se evalúa la regla en SAVH (experimento separado) se mantendría aparte, en `ocr/authorized_plates_simulada.csv`, y no reemplaza a esta. Las placas se pasan al formato XXX-0000 con `format_plate`, así que `GBA-1234`, `gba1234` y `GBA 1234` cuentan como la misma, igual que `GMS-429` y `GMS-0429`.


In [ ]:
# 3.5 · Lista de placas autorizadas. format_plate y clean_plate_text están en la normalización (2.1).


def load_authorized_plates(path: Path) -> set:
    """Placas del CSV en formato XXX-0000; si el archivo no existe, lo crea vacío."""
    path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists():
        path.write_text("")
        print(f"Aviso: {path} no existía; se creó vacío. Agrega placas autorizadas, una por línea.")
    with open(path, newline="") as f:
        plates = {format_plate(plate) for row in csv.reader(f) if row and row[0].strip() for plate,valid in [normalize_plate(row[0])] if valid}  # XXX-0000
    return plates - {"PLACA", "PLATE"}  # por si el CSV trae encabezado


def edit_distance_leq_1(a: str, b: str) -> bool:
    """True si las placas están a un carácter de distancia (Levenshtein ≤ 1): posible lectura de una autorizada."""
    if abs(len(a) - len(b)) > 1:
        return False
    return Levenshtein.distance(a, b) <= 1


AUTHORIZED_PLATES = load_authorized_plates(AUTHORIZED_LIST_PATH)
print(f"{len(AUTHORIZED_PLATES)} placas autorizadas en {AUTHORIZED_LIST_PATH}")

### 3.6 Regla de decisión del experimento

La lista CSV se usa únicamente en este notebook. Confianza baja, formato inválido o
una coincidencia aproximada pasan a revisión manual. Ni el SRI ni el nombre del
propietario autorizan un acceso. Este experimento **no acciona una barrera** ni escribe
operaciones en la aplicación.


In [ ]:
def decidir_ecuador(reading):
    plate=reading.plate
    if not plate or not math.isfinite(reading.det_confidence) or not math.isfinite(reading.ocr_confidence) or reading.det_confidence<DETECTOR_THRESHOLD or reading.ocr_confidence<OCR_THRESHOLD:
        return 'VERIFICACION_MANUAL'
    if plate in AUTHORIZED_PLATES:return 'AUTORIZADO'
    if any(edit_distance_leq_1(plate,p) for p in AUTHORIZED_PLATES):return 'VERIFICACION_MANUAL'
    return 'NO_REGISTRADO'


## 6. Sistema completo: pipeline de demostración en video
### 6.1 Fuente de video

El video de demostración existente es
[el archivo de Drive](https://drive.google.com/file/d/1NWB5XcewwXKMdBpenRZA_-gmrwTqnVBm/view).
Es una demostración, no una fuente confirmada de escenas ecuatorianas para entrenar.
Para Ecuador sustituye `VIDEO_PATH` por tu propia grabación; no se descarga nada hasta
activar `RUN_VIDEO` y `DOWNLOAD_DEMO_VIDEO`. No se usa como test independiente ni se
transforma automáticamente en datos de entrenamiento.


In [ ]:
VIDEO_ID='1NWB5XcewwXKMdBpenRZA_-gmrwTqnVBm'
VIDEO_PATH=WORK_DIR/'videos/demo_drive.mp4'
DOWNLOAD_DEMO_VIDEO=False
if RUN_VIDEO and DOWNLOAD_DEMO_VIDEO and not VIDEO_PATH.is_file():
    import gdown
    VIDEO_PATH.parent.mkdir(parents=True,exist_ok=True)
    downloaded=gdown.download(id=VIDEO_ID,output=str(VIDEO_PATH),quiet=False)
    if not downloaded or not VIDEO_PATH.is_file():raise FileNotFoundError('No se pudo descargar el video de demostración.')


### 6.2 Registro y muestreo

Se analiza un cuadro cada 2,5 segundos, **máximo ocho cuadros** por defecto. Todas las
placas que detecta el pipeline se guardan con país, modelo y confianza; no se fabrican
transcripciones de las placas ilegibles. Los CSV permanecen en `WORK_DIR/registros_csv/`.


In [ ]:
VIDEO_SAMPLE_SECONDS=2.5
MAX_VIDEO_FRAMES=8
VIDEO_RESULTS_PATH=WORK_DIR/'decisions_rtdetr_ecuador.csv'


### 6.3 Autorizaciones CSV y verificación en el SRI

El pipeline obtiene las autorizaciones exclusivamente de `authorized_plates.csv`. La lista de placas autorizadas se mantiene en `authorized_plates.csv`, en la carpeta de trabajo local, una placa por línea. Si el archivo no existe, se crea vacío. Las placas se normalizan al formato `XXX-0000`; una placa autorizada se marca `AUTORIZADO`, una coincidencia a un carácter pasa a `VERIFICACION_MANUAL` y el resto queda `NO_REGISTRADO`.

Las lecturas, consultas SRI y consultas del nombre público se guardan en archivos CSV bajo `registros_csv/` en el espacio temporal de la sesión. La sincronización externa permanece desactivada. Esta integración no escribe CSV operativos en GitHub ni cambia las autorizaciones de la aplicación.

**SRI opcional.** Con `SRI_VERIFICAR=True`, cada placa válida se consulta en segundo plano con el servicio público de la página «Valores a pagar por placa»: marca, modelo, año, país, último año pagado y valores pendientes. Si la placa antigua con cero inicial no aparece, reintenta con sus tres dígitos. Como máximo se envía una consulta cada 2 segundos y se reutiliza una consulta exitosa durante 24 horas. Los errores se guardan en `registros_csv/consultas_sri.csv` para poder reintentar en otra corrida. El SRI no devuelve el nombre del propietario y su respuesta no cambia la decisión de acceso.

Los CSV de trabajo quedan en `WORK_DIR/registros_csv/` durante la sesión y no se publican automáticamente. La sección 0 no monta Google Drive.


In [ ]:
# ===================== 6.3 · Archivos CSV + verificación automática en el SRI =====================
import base64
import csv
import datetime as dt
import io
import json
import os
import re
import threading
import time
import unicodedata
import uuid
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import quote

import requests

# SIRAP guarda la lista autorizada y los resultados del pipeline en archivos CSV.
# Los CSV de trabajo quedan en el espacio temporal de la sesión. Hemos desactivado por completo la sincronización con GitHub.
PIPELINE_CSV_DIR = WORK_DIR / "registros_csv"
PIPELINE_CSV_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATHS = {
    "lecturas": PIPELINE_CSV_DIR / "lecturas.csv",
    "consultas_sri": PIPELINE_CSV_DIR / "consultas_sri.csv",
    "propietarios_vehiculo": PIPELINE_CSV_DIR / "propietarios.csv",
    "historial_propietarios": PIPELINE_CSV_DIR / "historial_propietarios.csv",
    "consultas_propietario": PIPELINE_CSV_DIR / "consultas_propietario.csv",
}

SRI_VERIFICAR = False  # activar solo cuando quieras consultar el servicio
SRI_CONSULTAR_RUBROS = True
SRI_CACHE_HOURS = 24
SRI_MIN_INTERVAL_S = 2.0
SRI_TIMEOUT_S = 15
SRI_REST = "https://srienlinea.sri.gob.ec/sri-matriculacion-vehicular-recaudacion-servicio-internet/rest"
SRI_HEADERS = {"Accept": "application/json", "User-Agent": "ProyectoIntegrador-ALPR/1.0 (uso academico)"}



def now_ec() -> dt.datetime:
    """Hora de Ecuador (UTC-5), sin zona horaria para mantener el formato de los registros."""
    return dt.datetime.now(dt.timezone(dt.timedelta(hours=-5))).replace(tzinfo=None, microsecond=0)


def ascii_text(text) -> str:
    """OpenCV no dibuja tildes: 'LOGÍSTICA ÑANDÚ' → 'LOGISTICA NANDU'."""
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


class CsvPipelineStore:
    """Persistencia en archivos CSV del espacio temporal de Colab."""

    def __init__(self):
        self.lock = threading.RLock()
        for table, path in CSV_PATHS.items():
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text("", encoding="utf-8")

    def read(self, table: str) -> pd.DataFrame:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        path = CSV_PATHS[table]
        if not path.exists() or path.stat().st_size == 0:
            return pd.DataFrame()
        try:
            return pd.read_csv(path, keep_default_na=False)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

    def _write_rows(self, table: str, rows: list[dict]) -> None:
        path = CSV_PATHS[table]
        columns = list(dict.fromkeys([key for row in rows for key in row]))
        temp = path.with_suffix(path.suffix + ".tmp")
        with temp.open("w", encoding="utf-8-sig", newline="") as stream:
            if columns:
                writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore", lineterminator="\n")
                writer.writeheader()
                writer.writerows(rows)
        os.replace(temp, path)

    def insert(self, table: str, row: dict) -> None:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        with self.lock:
            current = self.read(table).to_dict("records")
            current.append({key: "" if value is None else value for key, value in row.items()})
            self._write_rows(table, current)

    def replace(self, table: str, rows: list[dict]) -> None:
        if table not in CSV_PATHS:
            raise ValueError(f"CSV desconocido: {table}")
        with self.lock:
            self._write_rows(table, rows)

    def latest_consulta(self, placa: str, max_age_hours: float):
        rows = self.read("consultas_sri")
        if rows.empty or "placa" not in rows or "consultado_en" not in rows:
            return None
        rows = rows[rows["placa"].astype(str).eq(placa)].copy()
        rows["_fecha"] = pd.to_datetime(rows["consultado_en"], errors="coerce")
        cutoff = now_ec() - dt.timedelta(hours=max_age_hours)
        rows = rows[rows["_fecha"].notna() & (rows["_fecha"] >= cutoff)]
        if "error" in rows:
            rows = rows[rows["error"].astype(str).eq("")]
        if rows.empty:
            return None
        record = rows.sort_values("_fecha").iloc[-1].drop(labels=["_fecha"]).to_dict()
        for key in ("encontrado", "matricula_al_dia"):
            if key in record and isinstance(record[key], str):
                record[key] = record[key].strip().lower() in ("true", "1", "sí", "si")
        for key in ("codigo_vehiculo", "ultimo_anio_pagado", "rubros_pendientes"):
            if key in record:
                try:
                    record[key] = int(float(record[key]))
                except (TypeError, ValueError):
                    record[key] = None
        for key in ("total_pendiente",):
            if key in record:
                try:
                    record[key] = float(record[key])
                except (TypeError, ValueError):
                    record[key] = None
        return record

    def owner(self, placa: str):
        rows = self.read("propietarios_vehiculo")
        if rows.empty or "placa" not in rows:
            return None
        matches = rows[rows["placa"].astype(str).eq(placa)]
        return None if matches.empty else matches.iloc[-1].to_dict()

    def owner_rows(self) -> list[dict]:
        return self.read("propietarios_vehiculo").to_dict("records")

    def upsert_owner(self, placa: str, nombre: str, fuente: str, fecha) -> tuple[str, str | None]:
        with self.lock:
            rows = self.read("propietarios_vehiculo").to_dict("records")
            existing = next((row for row in rows if str(row.get("placa")) == placa), None)
            anterior = existing.get("nombre") if existing else None
            if existing is None:
                rows.append({"placa": placa, "nombre": nombre, "fuente": fuente,
                             "consultado_en": fecha, "actualizado_en": fecha})
                estado = "insertado"
            elif anterior != nombre:
                existing.update({"nombre": nombre, "fuente": fuente,
                                 "consultado_en": fecha, "actualizado_en": fecha})
                estado = "actualizado"
            else:
                existing.update({"fuente": fuente, "consultado_en": fecha})
                estado = "sin_cambios"
            self._write_rows("propietarios_vehiculo", rows)
            if estado != "sin_cambios":
                self.insert("historial_propietarios", {"placa": placa, "cambiado_en": fecha,
                             "anterior": anterior, "nuevo": nombre, "fuente": fuente})
            return estado, anterior

    def clear_readings(self, corrida: str, video: str) -> None:
        rows = self.read("lecturas")
        if rows.empty:
            return
        keep = rows[~((rows.get("corrida", "").astype(str) == corrida)
                      & (rows.get("video", "").astype(str) == video))]
        self.replace("lecturas", keep.to_dict("records"))

    def verified_readings(self) -> pd.DataFrame:
        reads = self.read("lecturas")
        if reads.empty:
            return reads
        sri = self.read("consultas_sri")
        owners = self.read("propietarios_vehiculo")
        if not sri.empty and "placa" in sri:
            sri = sri.sort_values("consultado_en").drop_duplicates("placa", keep="last")
            rename = {key: f"sri_{key}" for key in sri.columns if key != "placa"}
            reads = reads.merge(sri.rename(columns=rename), on="placa", how="left")
        if not owners.empty and "placa" in owners:
            owners = owners.drop_duplicates("placa", keep="last").rename(columns={
                "nombre": "propietario_nombre", "fuente": "propietario_fuente",
                "consultado_en": "propietario_consultado_en", "actualizado_en": "propietario_actualizado_en"})
            reads = reads.merge(owners, on="placa", how="left")
        reads["registrado_csv"] = reads.get("autorizado_por_csv", False)
        return reads

    def save(self) -> None:
        """Conserva los CSV en el espacio temporal de Colab (sin sincronización con GitHub)."""
        try:
            self._sync_github()
        except Exception as error:
            print(f"CSV guardados en la carpeta temporal de la sesión; falló la sincronización con GitHub: {type(error).__name__}: {error}")

    def _sync_github(self) -> None:
        # Sincronización desactivada explícitamente a petición del usuario
        return


# La única fuente de placas autorizadas es el archivo CSV de excepciones/lista blanca.
# Las coincidencias a un carácter las manda a VERIFICACION_MANUAL en decide_access.
_csv_plates = load_authorized_plates(AUTHORIZED_LIST_PATH)
AUTHORIZED_PLATES = set(_csv_plates)
CSV_AUTORIZADAS = set(AUTHORIZED_PLATES)


def panel_detail(placa: str):
    if not placa:
        return None
    planta = "autorizado por CSV" if placa in AUTHORIZED_PLATES else "sin autorización en CSV"
    sri_text = sri.short(placa)
    return f"{planta} | {sri_text}" if sri_text else planta

_sri_session = requests.Session()
_sri_session.headers.update(SRI_HEADERS)


def _sri_get(url: str, params: dict):
    """GET con 2 reintentos si no hay conexión o el servidor falla (5xx)."""
    for attempt in range(3):
        try:
            response = _sri_session.get(url, params=params, timeout=SRI_TIMEOUT_S)
            if response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}", response=response)
            if response.status_code in (204, 404):
                return None, response.text
            response.raise_for_status()
            text = response.text.strip()
            return (json.loads(text) if text else None), text
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as error:
            is_server_error = error.response is None or error.response.status_code >= 500
            if attempt == 2 or not is_server_error:
                raise
            time.sleep(2 * (attempt + 1))


def consultar_sri(placa: str) -> dict:
    """Consulta el servicio público del SRI; no devuelve datos del propietario."""
    url = f"{SRI_REST}/BaseVehiculo/obtenerPorNumeroPlacaOPorNumeroCampvOPorNumeroCpn"
    found = lambda value: isinstance(value, dict) and bool(value.get("numeroPlaca") or value.get("codigoVehiculo"))
    compacta = placa.replace("-", "")
    vehiculo, texto = _sri_get(url, {"numeroPlacaCampvCpn": compacta})
    if not found(vehiculo) and re.fullmatch(r"[A-Z]{3}0[0-9]{3}", compacta):
        time.sleep(1)
        vehiculo, texto = _sri_get(url, {"numeroPlacaCampvCpn": compacta[:3] + compacta[4:]})
    resultado = {"placa": placa, "encontrado": False, "respuesta_vehiculo": texto[:4000] or None}
    if not found(vehiculo):
        resultado["alerta"] = "placa no encontrada en el SRI (¿lectura errónea del OCR?)"
        return resultado
    try:
        ultimo = int(vehiculo.get("ultimoAnioPagado"))
    except (TypeError, ValueError):
        ultimo = None
    resultado.update(
        encontrado=True, codigo_vehiculo=vehiculo.get("codigoVehiculo"), camv_cpn=vehiculo.get("numeroCamvCpn"),
        marca=vehiculo.get("descripcionMarca"), modelo=vehiculo.get("descripcionModelo"),
        anio_modelo=vehiculo.get("anioAuto"), pais=vehiculo.get("descripcionPais"), clase=vehiculo.get("nombreClase"),
        servicio=vehiculo.get("descripcionServicio"), canton=vehiculo.get("descripcionCanton"),
        estado_exoneracion=vehiculo.get("estadoExoneracion"), ultimo_anio_pagado=ultimo,
        matricula_al_dia=None if ultimo is None else ultimo >= now_ec().year,
        prohibido_enajenar=None if vehiculo.get("prohibidoEnajenar") is None else str(vehiculo["prohibidoEnajenar"]))
    alertas = [] if resultado["matricula_al_dia"] is not False else [f"sin pago del año {now_ec().year} (último: {ultimo})"]
    if SRI_CONSULTAR_RUBROS and vehiculo.get("codigoVehiculo"):
        try:
            rubros, texto_rubros = _sri_get(f"{SRI_REST}/ConsultaRubros/obtenerPorCodigoVehiculo",
                                            {"codigoVehiculo": vehiculo["codigoVehiculo"]})
            resultado["respuesta_rubros"] = texto_rubros[:4000] or None
            if rubros is None or isinstance(rubros, list):
                items = rubros or []
                resultado["rubros_pendientes"] = len(items)
                valores = [item.get("valor") for item in items if isinstance(item, dict)]
                if items and len(valores) == len(items) and all(isinstance(value, (int, float)) for value in valores):
                    resultado["total_pendiente"] = float(sum(valores))
                if items:
                    alertas.append("1 valor pendiente de pago" if len(items) == 1 else f"{len(items)} valores pendientes de pago")
        except Exception as error:
            resultado["respuesta_rubros"] = f"error: {type(error).__name__}: {error}"[:500]
    resultado["alerta"] = "; ".join(alertas) or None
    return resultado


class SRIVerifier:
    """Consulta el SRI en segundo plano y guarda la respuesta en CSV."""

    def __init__(self, store_: CsvPipelineStore):
        self.store = store_
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="sri")
        self.results, self.futures, self.lock, self.last_call = {}, {}, threading.Lock(), 0.0

    def submit(self, placa: str) -> None:
        if not SRI_VERIFICAR or not placa:
            return
        with self.lock:
            if placa in self.results or placa in self.futures:
                return
            cached = self.store.latest_consulta(placa, SRI_CACHE_HOURS)
            if cached is not None:
                self.results[placa] = cached
                return
            self.futures[placa] = self.pool.submit(self._consultar, placa)

    def _consultar(self, placa: str) -> dict:
        espera = SRI_MIN_INTERVAL_S - (time.monotonic() - self.last_call)
        if espera > 0:
            time.sleep(espera)
        self.last_call = time.monotonic()
        try:
            resultado = consultar_sri(placa)
        except Exception as error:
            resultado = {"placa": placa, "encontrado": None, "error": f"{type(error).__name__}: {error}"[:500]}
        resultado["consultado_en"] = now_ec()
        self.store.insert("consultas_sri", resultado)
        with self.lock:
            self.results[placa] = resultado
            self.futures.pop(placa, None)
        return resultado

    def wait(self) -> None:
        with self.lock:
            jobs = list(self.futures.values())
        for future in jobs:
            future.result()

    def summary(self, placa: str):
        if not SRI_VERIFICAR or not placa:
            return None
        result = self.results.get(placa)
        if result is None:
            return "SRI: consultando..." if placa in self.futures else None
        if result.get("error"):
            return "SRI: sin respuesta"
        if not result.get("encontrado"):
            return "SRI: placa no encontrada"
        vehicle = ascii_text(f"{result.get('marca') or ''} {result.get('modelo') or ''}").strip()[:26]
        estado = "al dia" if result.get("matricula_al_dia") else f"ultimo pago {result.get('ultimo_anio_pagado')}"
        return f"{vehicle} {result.get('anio_modelo') or ''} | {estado}"

    def short(self, placa: str):
        if not SRI_VERIFICAR or not placa:
            return None
        result = self.results.get(placa)
        if result is None:
            return "SRI consultando..." if placa in self.futures else None
        if result.get("error"):
            return "SRI sin respuesta"
        if not result.get("encontrado"):
            return "SRI: placa no encontrada"
        marca = ascii_text(result.get("marca") or "").strip()[:14]
        estado = "al dia" if result.get("matricula_al_dia") else f"ult. pago {result.get('ultimo_anio_pagado')}"
        return f"SRI {marca} {estado}".replace("  ", " ")


def guardar_lectura(reading,frame_index,seconds,corrida,video):
    """Registro local de RT-DETR Ecuador; no hay campos ni modelos YOLO."""
    store.insert('lecturas',dict(event_id=str(uuid.uuid4()),registrado_en=now_ec(),
        modelo=reading.detector,country='ecuador',corrida=corrida,video=Path(video).name,
        frame_index=frame_index,segundo_video=seconds,placa=reading.plate or '',
        texto_ocr=reading.plate_raw,formato_valido=bool(reading.plate),
        ocr_conf=reading.ocr_confidence,det_conf=reading.det_confidence,
        estado_acceso=decidir_ecuador(reading),autorizado_por_csv=reading.plate in AUTHORIZED_PLATES))


store = CsvPipelineStore()
sri = SRIVerifier(store)
print(f"Autorizaciones leídas de {AUTHORIZED_LIST_PATH.name}: {len(AUTHORIZED_PLATES)}")
print("Lecturas y consultas guardadas en CSV de la sesión:", PIPELINE_CSV_DIR)


### 6.4 Nombre del propietario (consulta pública opcional y registro local)

Esta parte conserva la integración del adjunto para consultar el formulario público de [EcuadorLegalOnline](https://www.ecuadorlegalonline.com/consultas/agencia-nacional-de-transito/consultar-a-quien-pertenece-un-vehiculo-por-placa-ant/) **solo si el proveedor permite esa consulta y el nombre está visible sin pagar**. No se comprobó aquí que el sitio siga ofreciendo ese dato; si está restringido, se mantiene el registro local. No abre «VER REPORTE COMPLETO» ni intenta obtener cédula o RUC, dirección o contacto. Si hay CAPTCHA, restricción, datos ocultos o errores, **no cambia** el dato que ya estaba guardado, y antes de guardar un nombre comprueba que la placa de la respuesta sea la que se consultó.

**Cómo se integra:** en el pipeline de video (6.6), cada placa leída con formato válido se consulta en segundo plano en el SRI y aquí, cada servicio en su propio hilo (los propietarios, de uno en uno). Se consulta también cuando la placa no aparece en la lista autorizada. La placa se normaliza a `XXX-0000`. Los resultados y cambios se guardan en CSV bajo `WORK_DIR/registros_csv/` durante la sesión:

- `propietarios.csv`: un registro vigente por placa. Si no existía, se inserta; si el nombre cambió, se actualiza en ese CSV cuando se consulta; si coincide, solo se actualiza `consultado_en`.
- `historial_propietarios.csv`: guarda el nombre anterior y el nuevo cuando hay un alta o un cambio real.
- `consultas_propietario.csv`: estado e incidencias de cada consulta.
- `lecturas.csv`, `consultas_sri.csv` y `propietarios.csv`: archivos que se combinan al generar el resumen de la corrida.

**Estado inicial:** `PROPIETARIO_WEB_ACTIVO = False`. Activarlo con `True` (celda siguiente) instala Playwright y Chromium, que necesitan acceso a Internet en Colab. Ponlo en `False` si el sitio no permite o restringe la automatización: el notebook no elude bloqueos ni pagos. `PROPIETARIO_CACHE_HORAS = 0` vuelve a consultar en cada corrida para detectar cambios. Cada resultado se escribe inmediatamente en su CSV temporal de la sesión; la sincronización con GitHub está desactivada y los CSV no se publican automáticamente.

**Consultas manuales:** `consultar_propietario_ahora('ABC0123')` consulta una placa y `consultar_propietarios_registrados(limite=10)` consulta los nombres de las placas que aparecen en `authorized_plates.csv` y todavía no tienen resultado guardado. Entre peticiones al sitio se esperan 12 s (`PROPIETARIO_INTERVALO_S`).


In [ ]:
# 6.4 · Navegador (Playwright + Chromium) para leer SOLO el nombre público del propietario.
# Ponlo en False si el sitio no permite consultas automáticas o deja de responder. Nunca se resuelven CAPTCHAs,
# se hacen pagos ni se saltan controles del sitio.
PROPIETARIO_WEB_ACTIVO = False

if PROPIETARIO_WEB_ACTIVO:
    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("playwright") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    # «playwright install» no repite la descarga si Chromium ya está en la sesión.
    try:
        subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
        print("Navegador Playwright preparado para la consulta pública de nombre.")
    except subprocess.CalledProcessError as exc:
        PROPIETARIO_WEB_ACTIVO = False
        print("Consulta de propietarios desactivada (no fue posible preparar Chromium):", exc)


In [ ]:
# ===================== 6.4 · Nombre público del propietario y archivos CSV =====================
# Fuente: solo el campo «Nombre», si el proveedor lo permite y está visible sin pagar. No se abre el reporte completo ni se
# guardan cédulas, direcciones o teléfonos. PROPIETARIO_WEB_ACTIVO se fija en la celda anterior.
PROPIETARIO_CACHE_HORAS = 0       # 0 = volver a consultar en cada corrida (detecta cambios de dueño)
PROPIETARIO_INTERVALO_S = 12.0    # espera mínima entre consultas al sitio, de una en una
PROPIETARIO_TIMEOUT_MS = 25000    # tiempo máximo para cargar la página y recibir la respuesta

PROPIETARIO_URL = (
    "https://www.ecuadorlegalonline.com/consultas/agencia-nacional-de-transito/"
    "consultar-a-quien-pertenece-un-vehiculo-por-placa-ant/"
)

def placa_consulta_propietario(placa: str) -> str:
    """Placa en formato XXX-0000 para consultar y guardar; ValueError si no es una placa ecuatoriana válida."""
    compacta = re.sub(r"[^A-Z0-9]", "", str(placa).strip().upper())
    if not re.fullmatch(r"[A-Z]{3}[0-9]{3,4}", compacta):
        raise ValueError(f"Formato de placa inválido: {placa!r}")
    return compacta[:3] + "-" + compacta[3:].zfill(4)


def _nombre_visible(texto: str | None) -> str | None:
    """El nombre si es un texto visible y plausible; None si está vacío, oculto (xxxx), es otro rótulo o es demasiado largo."""
    if not texto:
        return None
    valor = re.sub(r"\s+", " ", str(texto)).strip()
    if (not valor or len(valor) > 250 or not re.search(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", valor)
            or re.fullmatch(r"[xX*\- .]+", valor)
            or re.match(r"^(?:c[eé]dula|ruc|direcci[oó]n|tel[eé]fono|due[nñ]o anterior|disponible|no disponible|oculto|ninguno|nombre$)", valor, re.I)):
        return None
    return valor


def extraer_nombre_publico(texto: str) -> str | None:
    """Lee solo la línea «Nombre» del bloque «Datos del propietario del vehículo» que muestra la página."""
    zona = re.search(r"DATOS DEL PROPIETARIO DEL VEH[IÍ]CULO(.*?)(?:VER REPORTE COMPLETO|¿No encuentra datos|$)",
                     texto, re.I | re.S)
    if not zona:
        return None
    lineas = [l.strip() for l in zona.group(1).splitlines() if l.strip()]
    for i, linea in enumerate(lineas):
        m = re.match(r"^Nombre\s*[:：]?\s*(.*)$", linea, re.I)
        if m:
            valor = m.group(1).strip() or (lineas[i + 1] if i + 1 < len(lineas) else "")
            return _nombre_visible(valor)
    return None


def _placa_de_resultado(page) -> str | None:
    """La placa que la página muestra como resultado («Placa consultada»), no la que se escribió en el buscador.

    Así no se guarda el nombre de un resultado de ejemplo, de una consulta anterior o de otra placa.
    """
    body = page.locator("body").inner_text(timeout=8000)
    m = re.search(r"Placa consultada\s*[:：]?\s*([A-Z]{3}[-\s]?[0-9]{3,4})\b", body, re.I)
    if m:
        return placa_consulta_propietario(m.group(1))
    # En algunas versiones de la página la placa va en un campo de solo lectura junto al rótulo «Placa consultada»,
    # que no aparece en el texto de la página.
    try:
        bloque = page.get_by_text("Placa consultada", exact=True).first.locator("xpath=..")
        for candidato in bloque.locator("input[readonly], input[disabled]").all():
            valor = candidato.input_value()
            try:
                return placa_consulta_propietario(valor)
            except ValueError:
                pass
    except Exception:
        pass
    return None


def consultar_nombre_publico_web(placa: str) -> dict:
    """Llena el formulario público con la placa y lee solo el nombre gratuito → {"estado", "nombre" o "detalle"}.

    No toca «VER REPORTE COMPLETO», CAPTCHAs, servicios privados ni pagos. Si no puede confirmar que la respuesta es de
    la placa consultada, no devuelve nombre.
    """
    from playwright.sync_api import sync_playwright
    placa = placa_consulta_propietario(placa)
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True)
        try:
            page = browser.new_page(locale="es-EC", viewport={"width": 1360, "height": 900})
            page.goto(PROPIETARIO_URL, wait_until="domcontentloaded", timeout=PROPIETARIO_TIMEOUT_MS)
            entrada = page.locator('input[name*="placa" i], input[placeholder*="placa" i]')
            if entrada.count() == 0:
                entrada = page.get_by_label("Placa", exact=True)
            if entrada.count() == 0:
                return {"estado": "interfaz_no_reconocida", "detalle": "No se localizó el campo Placa"}
            entrada.first.fill(placa.replace("-", ""))
            consultar = page.get_by_role("button", name=re.compile(r"^Consultar$", re.I))
            if consultar.count() == 0:
                return {"estado": "interfaz_no_reconocida", "detalle": "No se localizó Consultar"}
            consultar.first.click(timeout=PROPIETARIO_TIMEOUT_MS)
            fin = time.monotonic() + PROPIETARIO_TIMEOUT_MS / 1000
            while time.monotonic() < fin:
                texto = page.locator("body").inner_text(timeout=8000)
                if re.search(r"captcha|acceso denegado|verifica que (eres|no eres)", texto, re.I):
                    return {"estado": "acceso_restringido", "detalle": "El sitio requiere verificación humana"}
                respondida = _placa_de_resultado(page)
                if respondida:
                    if respondida != placa:
                        return {"estado": "placa_no_coincide", "detalle": "La placa de la respuesta no coincide"}
                    nombre = extraer_nombre_publico(texto)
                    if nombre:
                        return {"estado": "encontrado", "nombre": nombre}
                    if re.search(r"no se encontr[oó]|sin resultados|veh[ií]culo no existe", texto, re.I):
                        return {"estado": "sin_datos", "detalle": "El sitio no devolvió propietario"}
                time.sleep(0.75)
            return {"estado": "sin_respuesta_verificable", "detalle": "No se confirmó placa y nombre visibles"}
        finally:
            browser.close()


def sincronizar_nombre_propietario(placa: str, nombre: str, fuente="ecuadorlegalonline_publico") -> str:
    """Guarda el nombre vigente y conserva cambios anteriores en archivos CSV."""
    placa = placa_consulta_propietario(placa)
    nombre = _nombre_visible(nombre)
    if not nombre:
        raise ValueError("Nombre vacío, oculto o inválido; no se modifica el CSV")
    estado, _ = store.upsert_owner(placa, nombre, fuente, now_ec())
    return estado


def registrar_consulta_propietario(placa, estado, detalle=None):
    """Una fila en consultas_propietario por cada consulta, con su estado (encontrado, sin_datos, error…)."""
    store.insert("consultas_propietario", {
        "placa": placa_consulta_propietario(placa), "consultado_en": now_ec(),
        "estado": estado, "detalle": str(detalle)[:300] if detalle else None,
        "fuente": "ecuadorlegalonline_publico"
    })


class PropietarioVerifier:
    """Consulta los propietarios en un hilo aparte, de uno en uno, para no frenar el video ni el SRI."""
    def __init__(self, store_: CsvPipelineStore):
        self.store = store_
        self.pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="propietario")
        self.lock = threading.Lock()
        self.results, self.futures = {}, {}
        self.last_call = 0.0

    def nueva_corrida(self):
        """Al empezar una corrida olvida los resultados en memoria, para volver a consultar las placas."""
        self.wait()
        with self.lock:
            self.results.clear()

    def submit(self, placa: str):
        """Pone la placa en la cola si es válida y no se consultó ya en esta corrida; devuelve su tarea o None."""
        if not PROPIETARIO_WEB_ACTIVO:
            return None
        try:
            placa = placa_consulta_propietario(placa)
        except ValueError:
            return None
        with self.lock:
            if placa in self.futures or placa in self.results:
                return self.futures.get(placa)
            future = self.pool.submit(self._consultar, placa)
            self.futures[placa] = future
            return future

    def _consultar(self, placa: str):
        """Consulta una placa respetando el intervalo, guarda el nombre y registra la consulta."""
        resultado = {"estado": "error", "detalle": "No iniciada"}
        try:
            if PROPIETARIO_CACHE_HORAS > 0:
                reciente = self.store.owner(placa)
                if reciente:
                    consultado = pd.to_datetime(reciente.get("consultado_en"), errors="coerce")
                    if pd.notna(consultado) and consultado >= now_ec() - dt.timedelta(hours=PROPIETARIO_CACHE_HORAS):
                        resultado = {"estado": "en_cache"}
                        return resultado
            espera = PROPIETARIO_INTERVALO_S - (time.monotonic() - self.last_call)
            if espera > 0:
                time.sleep(espera)
            self.last_call = time.monotonic()
            resultado = consultar_nombre_publico_web(placa)
            if resultado["estado"] == "encontrado":
                resultado["accion_csv"] = sincronizar_nombre_propietario(placa, resultado["nombre"])
            registrar_consulta_propietario(placa, resultado["estado"], resultado.get("detalle"))
            # El CSV local se escribe al registrar el nombre; GitHub se sincroniza al cerrar el pipeline.
        except Exception as exc:
            resultado = {"estado": "error", "detalle": f"{type(exc).__name__}: {exc}"[:300]}
            try:
                registrar_consulta_propietario(placa, "error", resultado["detalle"])
            except Exception:
                pass
        finally:
            with self.lock:
                self.results[placa] = resultado
                self.futures.pop(placa, None)
        return resultado

    def wait(self):
        """Espera las consultas en curso. El diccionario cambia mientras terminan: se copia con el candado tomado."""
        with self.lock:
            trabajos = list(self.futures.values())
        for future in trabajos:
            future.result()


if "propietario" in globals() and getattr(propietario, "store", None) is not store:
    try:
        propietario.wait()
        propietario.pool.shutdown(wait=False)
    except Exception:
        pass
propietario = PropietarioVerifier(store)


def consultar_propietario_ahora(placa: str) -> dict:
    """Consulta una placa ahora (aunque ya se haya consultado en esta corrida) y devuelve el resultado ya guardado."""
    placa = placa_consulta_propietario(placa)
    if not PROPIETARIO_WEB_ACTIVO:
        local=store.owner(placa)
        return {"estado":"registro_local","nombre":local["nombre"]} if local else {"estado":"desactivado","detalle":"Sin propietario local; la consulta web es opcional."}
    # Para repetir la consulta se olvida el resultado en memoria.
    propietario.wait()
    with propietario.lock:
        propietario.results.pop(placa, None)
    tarea = propietario.submit(placa)
    return tarea.result() if tarea is not None else {"estado": "no_iniciado"}


def consultar_propietarios_registrados(limite: int | None = None):
    """Consulta el nombre de las placas de la planta que todavía no lo tienen (como máximo `limite`).

    Opcional: respeta el intervalo entre consultas al sitio, así que 10 placas tardan unos 2 minutos.
    """
    if not PROPIETARIO_WEB_ACTIVO:
        print("La consulta web está desactivada.")
        return pd.DataFrame()
    ya_consultadas = {str(row.get("placa", "")) for row in store.owner_rows()}
    placas = sorted(AUTHORIZED_PLATES - ya_consultadas)
    if limite is not None:
        placas = placas[:limite]
    trabajos = {p: propietario.submit(p) for p in placas}
    resultados = [{"placa": p, **(t.result() if t else {"estado": "no_iniciado"})} for p, t in trabajos.items()]
    return pd.DataFrame(resultados)


print("Propietarios: consulta del nombre público activa:", PROPIETARIO_WEB_ACTIVO)
print("Propietarios guardados en CSV:", len(store.owner_rows()))
print("Consultar placa puntual: consultar_propietario_ahora('ABC0123')")
print("Completar propietarios faltantes (opcional): consultar_propietarios_registrados(limite=10)")


### 6.5 Comprobación de la extracción del nombre (sin usar el sitio web)

Estas pruebas verifican, con textos de ejemplo, que solo se extrae el nombre de la línea visible «Nombre» y que los marcadores ocultos (`xxxxxx`) no se toman como propietario. **No** consultan el sitio ni modifican la base. Al procesar el video, las consultas externas solo se hacen si se activaron expresamente en 6.3/6.4. El SRI no aporta etiquetas ni propietarios para entrenar.


In [ ]:
# 6.5 · Comprobaciones locales sin abrir navegador, sin llamadas al proveedor y sin escribir datos externos.
_ejemplo_publico = """Placa consultada ABC0123
DATOS DEL PROPIETARIO DEL VEHÍCULO
Nombre
EMPRESA DE EJEMPLO
Cédula / RUC
🔒 Disponible
VER REPORTE COMPLETO"""
_ejemplo_oculto = """DATOS DEL PROPIETARIO DEL VEHÍCULO
Nombre xxxxxx xxxxxx xxxxxx
Cédula / RUC
VER REPORTE COMPLETO"""
assert extraer_nombre_publico(_ejemplo_publico) == "EMPRESA DE EJEMPLO"
assert extraer_nombre_publico(_ejemplo_oculto) is None
assert placa_consulta_propietario("ABC123") == "ABC-0123"
print("Correcto: la extracción local del texto de ejemplo devuelve el nombre y rechaza marcadores ocultos; no se consultó ningún sitio.")


In [ ]:
# 6.5 · Revisión de los archivos CSV de propietarios
for nombre, tabla in (("propietarios.csv", "propietarios_vehiculo"),
                      ("historial_propietarios.csv", "historial_propietarios"),
                      ("consultas_propietario.csv", "consultas_propietario")):
    datos = store.read(tabla)
    print(f"{nombre}: {len(datos)} registros")
    if not datos.empty:
        display(datos.head(10))


### 6.6 Ejecutar y guardar el flujo integrado

Activa `RUN_INFERENCE` y carga el modelo en 3.2; después activa `RUN_VIDEO`, elige la
ruta de un video propio y ejecuta esta celda. Mantiene un máximo de ocho cuadros.
Las consultas SRI/propietario son opcionales, están separadas del entrenamiento y no
alteran la decisión CSV. No se abre una barrera ni se publican los CSV.


In [ ]:
if RUN_VIDEO:
    import cv2
    if pipeline is None:raise RuntimeError('Activa RUN_INFERENCE y ejecuta 3.2 antes del video.')
    if not VIDEO_PATH.is_file():raise FileNotFoundError(f'Falta el video: {VIDEO_PATH}')
    cap=cv2.VideoCapture(str(VIDEO_PATH))
    if not cap.isOpened():raise ValueError('No se pudo abrir el video.')
    fps=cap.get(cv2.CAP_PROP_FPS) or 25
    step=max(1,round(fps*VIDEO_SAMPLE_SECONDS))
    corrida=str(uuid.uuid4());index=0;sampled=0;rows=[]
    try:
        while sampled<MAX_VIDEO_FRAMES:
            ok,frame=cap.read()
            if not ok:break
            if index%step==0:
                readings=pipeline.recognize_many(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
                if not readings:
                    rows.append(dict(frame_index=index,second=index/fps,country='ecuador',plate_pred='',status='SIN_DETECCION'))
                for reading in readings:
                    guardar_lectura(reading,index,index/fps,corrida,VIDEO_PATH)
                    rows.append(dict(frame_index=index,second=index/fps,country='ecuador',plate_pred=reading.plate,
                        det_confidence=reading.det_confidence,ocr_confidence=reading.ocr_confidence,
                        status=decidir_ecuador(reading)))
                    if reading.plate:sri.submit(reading.plate);propietario.submit(reading.plate)
                sampled+=1
            index+=1
    finally:cap.release()
    sri.wait();propietario.wait()
    pd.DataFrame(rows).to_csv(VIDEO_RESULTS_PATH,index=False)
    store.verified_readings().to_csv(PIPELINE_CSV_DIR/'lecturas_enriquecidas.csv',index=False)
    store.save()  # solo archivos temporales; sincronización externa desactivada
    print('Cuadros analizados:',sampled,'| predicciones:',VIDEO_RESULTS_PATH)
else:
    print('Video preparado; RUN_VIDEO=False. No hubo inferencia ni consultas externas.')
